In [1]:
!pip install -U "pandas" "indic-nlp-library" "transformers[torch]" "datasets" "httpx==0.24.0" "accelerate>=0.26.0" "scikit-learn"
!git clone https://github.com/anoopkunchukuttan/indic_nlp_resources.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.3/75.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3

Cloning into 'indic_nlp_resources'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (13/13), done.
^C


In [1]:
from indicnlp import common
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
import pandas as pd
from sklearn.model_selection import train_test_split
import re
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Set up IndicNLP
common.set_resources_path("./indic_nlp_resources")
factory = IndicNormalizerFactory()
normalizer = factory.get_normalizer("ta")

# Load and clean data (assuming your CSV is uploaded to Colab as 'Tamil-News-Headlines.csv')
df = pd.read_csv('Tamil-News-Headlines.csv', encoding='utf-8')  # Or use 'utf-8-sig' if BOM issue

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^\u0B80-\u0BFF\s.,!?]', '', text)
    return normalizer.normalize(text.strip()) if text.strip() else ""

df['News'] = df['News'].apply(clean_text)
df_train = df[['News', 'Authenticity']].copy()
df_train.columns = ['text', 'label']

# Split data (80/10/10)
train_df, temp_df = train_test_split(df_train, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 4180, Val: 523, Test: 523


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [13]:
# Model ID
model_id = "google/muril-base-cased"

# Training args (same as yours, with push to hub)
training_args = TrainingArguments(
    output_dir='./results_muril',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to=[],
    logging_strategy="epoch",
    save_total_limit=2,
    seed=42,
    fp16=True,  # GPU acceleration
    push_to_hub=True,  # Upload to HF Hub
    hub_model_id="AshokanK/tamil-fake-news-muril"  # Replace '<YOUR_HF_USERNAME>' with your actual HF username, e.g., "grok-user/tamil-fake-news-muril"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='binary')
    return {"accuracy": acc, "f1": f1}

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128  # Good for headlines
    )

# Prepare datasets
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize_function, batched=True)
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True)).map(tokenize_function, batched=True)
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize_function, batched=True)

train_ds = train_ds.rename_column("label", "labels").remove_columns(["text"])
val_ds = val_ds.rename_column("label", "labels").remove_columns(["text"])
test_ds = test_ds.rename_column("label", "labels").remove_columns(["text"])

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# Train and push to hub
trainer.train()
trainer.push_to_hub()  # Uploads the best model

# Evaluate on test set
test_results = trainer.evaluate(test_ds)
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f} | F1: {test_results['eval_f1']:.4f}")

# Explicitly save the best model and tokenizer locally after training
final_save_path = "./final_trained_model"
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"Best model and tokenizer explicitly saved to {final_save_path}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/4180 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Map:   0%|          | 0/523 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.382000,0.123006,0.969407,0.970370
2,0.105300,0.104093,0.977055,0.978102
3,0.045000,0.090988,0.980880,0.981618
4,0.015300,0.107434,0.980880,0.981685


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s_muril/training_args.bin: 100%|##########| 5.78kB / 5.78kB            

  ...s_muril/model.safetensors:   4%|3         | 33.5MB /  950MB            

Test Accuracy: 0.9828 | F1: 0.9843
Best model and tokenizer explicitly saved to ./final_trained_model


In [14]:
from transformers import pipeline

# Load your trained model from the new, explicitly saved local directory
classifier = pipeline("text-classification", model="./final_trained_model")

# Example Tamil headline (replace with any)
headline = "பாஸ்வேர்டை பகிரும் பயனர்களிடம் கூடுதல் கட்டணம்: நெட்ஃப்ளிக்ஸ் பலே திட்டம்"  # From your sample

result = classifier(headline)[0]
label = "Real" if result['label'] == 'LABEL_0' else "Fake" # Assuming 0 is Real and 1 is Fake
confidence = result['score']
print(f"Headline: {headline}\nPrediction: {label} (Confidence: {confidence:.4f})")

The tokenizer you are loading from './final_trained_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cuda:0


Headline: பாஸ்வேர்டை பகிரும் பயனர்களிடம் கூடுதல் கட்டணம்: நெட்ஃப்ளிக்ஸ் பலே திட்டம்
Prediction: Real (Confidence: 0.9933)


In [15]:
# This cell is no longer needed as we're explicitly saving the model and tokenizer in the training cell.
# To avoid future confusion, this cell can be deleted or commented out.
# If you need to reload the best model later, you can use:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# final_save_path = "./final_trained_model"
# tokenizer = AutoTokenizer.from_pretrained(final_save_path)
# model = AutoModelForSequenceClassification.from_pretrained(final_save_path)
